In [6]:
import signal
import wandb
import torch
import os 

import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from hydra import compose, initialize

from codefiles.helpers import is_running_in_notebook  # for reloading modules instead of restarting kernel
if is_running_in_notebook():
    from codefiles import helpers
    import importlib
    importlib.reload(helpers)
from codefiles.helpers import set_all_seeds, signal_handler, build_model, build_lightningmodule, build_datamodule

os.environ["WANDB_SILENT"] = "true"
torch.set_float32_matmul_precision("high")

def main(cfg) -> None:
    wandb.finish()
    set_all_seeds(seed=cfg.seed)
    wandb.init(
        project=cfg.wandb.project,
        group=None if cfg.wandb.group == "None" else cfg.wandb.group,
        config={key: value for key, value in cfg.items()},
    )

    model = build_model(cfg)
    lightningmodule = build_lightningmodule(cfg, model)
    datamodule = build_datamodule(cfg)

    trainer = pl.Trainer(
        logger=WandbLogger(project=cfg.wandb.project, dir="wandb/"),
        log_every_n_steps=1,
        accelerator='gpu',
        devices=1,
        max_epochs=cfg.max_epochs,
        precision=cfg.precision
    )

    trainer.fit(lightningmodule, datamodule)
    wandb.finish()

if __name__ == "__main__":
    CONFIG_NAME = "config"
    signal.signal(signal.SIGINT, signal_handler)
    with initialize(version_base="1.1", config_path="config"):
        cfg = compose(config_name=f"{CONFIG_NAME}")
    main(cfg)

Seed set to 420


TypeError: __init__() got an unexpected keyword argument 'seed'